# Train Resume NER Model with Class Weights

This notebook trains a DeBERTa model with class weights to fix the I- tag prediction issue.

**Before running:**
1. Runtime → Change runtime type → GPU (T4)
2. Upload your `data/` folder containing `dataset_train.conll` and `dataset_test.conll`

In [ ]:
# Install dependencies
!pip install transformers datasets evaluate seqeval torch accelerate -q

In [ ]:
# Upload your training data
# Click the folder icon on the left, then upload:
# - data/dataset_train.conll
# - data/dataset_test.conll

# Verify files exist
import os
if os.path.exists('data/dataset_train.conll'):
    print("✅ Training data found")
else:
    print("❌ Please upload data/dataset_train.conll")

if os.path.exists('data/dataset_test.conll'):
    print("✅ Test data found")
else:
    print("❌ Please upload data/dataset_test.conll")

In [ ]:
# Training script with class weights
import json
import torch
import numpy as np
from pathlib import Path
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torch.nn import CrossEntropyLoss
import evaluate

# Configuration
MODEL_NAME = "microsoft/deberta-v3-base"
OUTPUT_DIR = "./resume-ner-deberta-weighted"

CONFIG = {
    "num_train_epochs": 15,
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "warmup_steps": 1000,
    "weight_decay": 0.01,
    "i_tag_weight": 2.0,
    "max_length": 1024,
}

print("✅ Configuration loaded")
print(f"   Model: {MODEL_NAME}")
print(f"   Epochs: {CONFIG['num_train_epochs']}")
print(f"   I- tag weight: {CONFIG['i_tag_weight']}x")

In [ ]:
# Load CoNLL data
def read_conll_file(file_path):
    """Read CoNLL format file"""
    sentences = []
    sentence_tokens = []
    sentence_labels = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if sentence_tokens:
                    sentences.append({
                        'tokens': sentence_tokens,
                        'ner_tags': sentence_labels
                    })
                    sentence_tokens = []
                    sentence_labels = []
            else:
                parts = line.split()
                if len(parts) == 2:
                    token, label = parts
                    sentence_tokens.append(token)
                    sentence_labels.append(label)
    
    if sentence_tokens:
        sentences.append({
            'tokens': sentence_tokens,
            'ner_tags': sentence_labels
        })
    
    return sentences

print("📂 Loading training data...")
train_data = read_conll_file('data/dataset_train.conll')
test_data = read_conll_file('data/dataset_test.conll')

print(f"   Train samples: {len(train_data):,}")
print(f"   Test samples: {len(test_data):,}")

In [ ]:
# Get unique labels
all_labels = set()
for example in train_data + test_data:
    all_labels.update(example['ner_tags'])

label_list = sorted(all_labels, key=lambda x: (0 if x == 'O' else 1 if x.startswith('B-') else 2, x))
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"🏷️  Labels: {len(label_list)}")
print(f"   {label_list[:10]}...")

In [ ]:
# Compute class weights
label_counts = Counter()
for example in train_data:
    for label in example['ner_tags']:
        label_counts[label2id[label]] += 1

weights = torch.ones(len(label2id))
for label_id, label_name in id2label.items():
    if label_name.startswith('I-'):
        weights[label_id] = CONFIG['i_tag_weight']

print(f"⚖️  Class weights computed")
print(f"   I- tags get {CONFIG['i_tag_weight']}x weight")

In [ ]:
# Load tokenizer
print(f"🔤 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenize function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=CONFIG["max_length"],
        padding=False
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Convert to datasets format
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# Tokenize
print("⚙️  Tokenizing...")
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=test_dataset.column_names)

print("✅ Tokenization complete")

In [ ]:
# Load model
print(f"🤖 Loading model: {MODEL_NAME}")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("✅ Model loaded")

In [ ]:
# Metrics
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = []
    true_predictions = []
    
    for prediction, label in zip(predictions, labels):
        true_label = []
        true_prediction = []
        
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                true_label.append(id2label[label_id])
                true_prediction.append(id2label[pred_id])
        
        true_labels.append(true_label)
        true_predictions.append(true_prediction)
    
    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("✅ Metrics configured")

In [ ]:
# Custom trainer with class weights
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss_fct = CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100
        )
        
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

print("✅ Custom trainer configured")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    save_total_limit=3,
    fp16=True,
    report_to="none",
    push_to_hub=False
)

print("📊 Training configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   FP16: {training_args.fp16}")

In [ ]:
# Initialize trainer
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=weights
)

print("✅ Trainer initialized")

In [ ]:
# Train!
print("\n🏋️  Starting training...")
print("⏱️  Expected time: ~30-40 minutes on T4 GPU")
print("=" * 80)

trainer.train()

print("\n✅ Training complete!")

In [ ]:
# Evaluate
print("\n📊 Final evaluation...")
results = trainer.evaluate()

print(f"\n   Precision: {results['eval_precision']:.4f}")
print(f"   Recall: {results['eval_recall']:.4f}")
print(f"   F1 Score: {results['eval_f1']:.4f}")
print(f"   Accuracy: {results['eval_accuracy']:.4f}")

In [ ]:
# Save model
print(f"\n💾 Saving model to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save label mappings
label_mappings = {
    "labels": label_list,
    "label2id": label2id,
    "id2label": id2label
}

with open(f"{OUTPUT_DIR}/label_mappings.json", "w") as f:
    json.dump(label_mappings, f, indent=2)

print("✅ Model saved!")
print(f"\n📦 Download the '{OUTPUT_DIR}' folder to use the model locally")

In [ ]:
# Test the model
from transformers import pipeline

print("\n🧪 Testing model...")
ner = pipeline("ner", model=OUTPUT_DIR, aggregation_strategy="simple")

test_text = "I worked as a Junior Full Stack Developer at Google from Jan 2020 to Dec 2022"
result = ner(test_text)

print(f"\n📝 Input: {test_text}")
print("\n🔍 Results:")
for entity in result:
    print(f"   {entity['entity_group']:15s}: {entity['word']}")

print("\n✅ If you see 'Junior Full Stack Developer' as one entity, the model is working correctly!")